In [1]:
import tensorflow as tf
from tensorflow import keras
import os
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
data_dir = 'dataset'

In [3]:
img_height = 224
img_width = 224
batch_size = 32

In [4]:
class_names = ['NORMAL', 'PNEUMONIA']
num_classes = len(class_names)

In [5]:
# Data Augmentation
data_augmentation = tf.keras.Sequential([
  tf.keras.layers.RandomFlip("horizontal_and_vertical"),
  tf.keras.layers.RandomRotation(0.2),
])

In [6]:
def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [img_height, img_width])
    image = data_augmentation(image)  # Apply data augmentation
    image /= 255.0  # Normalize to [0,1]
    return image, label

In [7]:
def get_image_paths_and_labels(data_dir):
    image_paths = []
    labels = []
    for label, class_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, class_name)
        class_image_paths = [os.path.join(class_dir, fname) for fname in os.listdir(class_dir) if fname.endswith(('.jpg', '.jpeg', '.png'))]
        image_paths += class_image_paths
        labels += [label] * len(class_image_paths)
    return image_paths, labels

In [8]:
image_paths, labels = get_image_paths_and_labels(data_dir)
image_paths_train, image_paths_val, labels_train, labels_val = train_test_split(image_paths, labels, test_size=0.2, random_state=123)

In [9]:
train_ds = tf.data.Dataset.from_tensor_slices((image_paths_train, labels_train))
val_ds = tf.data.Dataset.from_tensor_slices((image_paths_val, labels_val))

In [10]:
train_ds = train_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(load_and_preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)

In [11]:
train_ds = train_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [12]:
# AlexNet Architecture
alexnet_model = keras.Sequential([
    keras.layers.Conv2D(filters=96, kernel_size=(11, 11), strides=(4, 4), activation='relu', input_shape=(img_height, img_width, 3)),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2)),

    keras.layers.Conv2D(filters=256, kernel_size=(5, 5), strides=(1, 1), padding='same', activation='relu'),
    keras.layers.BatchNormalization(),
    keras.layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2)),

    keras.layers.Conv2D(filters=384, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu'),
    keras.layers.Conv2D(filters=384, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu'),
    keras.layers.Conv2D(filters=256, kernel_size=(3, 3), strides=(1, 1), padding='same', activation='relu'),
    keras.layers.MaxPooling2D(pool_size=(3, 3), strides=(2, 2)),

    keras.layers.Flatten(),
    keras.layers.Dense(4096, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(4096, activation='relu'),
    keras.layers.Dropout(0.5),
    keras.layers.Dense(num_classes)
])

In [13]:
alexnet_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 54, 54, 96)        34944     
                                                                 
 batch_normalization (Batch  (None, 54, 54, 96)        384       
 Normalization)                                                  
                                                                 
 max_pooling2d (MaxPooling2  (None, 26, 26, 96)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 26, 26, 256)       614656    
                                                                 
 batch_normalization_1 (Bat  (None, 26, 26, 256)       1024      
 chNormalization)                                                
                                                      

In [14]:
alexnet_model.compile(optimizer=keras.optimizers.Adam(),
                      loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                      metrics=['accuracy'])

In [15]:
history = alexnet_model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=10
)

Epoch 1/10


147/147 [==============================] - 143s 960ms/step - loss: 1.1124 - accuracy: 0.7222 - val_loss: 0.6495 - val_accuracy: 0.7235
Epoch 2/10
147/147 [==============================] - 101s 687ms/step - loss: 0.4925 - accuracy: 0.7338 - val_loss: 1.9926 - val_accuracy: 0.7235
Epoch 3/10
147/147 [==============================] - 102s 691ms/step - loss: 0.4049 - accuracy: 0.8072 - val_loss: 0.9066 - val_accuracy: 0.2765
Epoch 4/10
147/147 [==============================] - 102s 692ms/step - loss: 0.4380 - accuracy: 0.7936 - val_loss: 0.3718 - val_accuracy: 0.8353
Epoch 5/10
147/147 [==============================] - 101s 689ms/step - loss: 0.3562 - accuracy: 0.8533 - val_loss: 0.3904 - val_accuracy: 0.8259
Epoch 6/10
147/147 [==============================] - 103s 702ms/step - loss: 0.3331 - accuracy: 0.8719 - val_loss: 0.3881 - val_accuracy: 0.8430
Epoch 7/10
147/147 [==============================] - 101s 687ms/step - loss: 0.3515 - accuracy: 0.8664 - val_loss: 1.2700

In [16]:
loss, accuracy = alexnet_model.evaluate(val_ds)
print(f"Validation accuracy: {accuracy*100:.2f}%")

37/37 [==============================] - 6s 155ms/step - loss: 0.3996 - accuracy: 0.8379
Validation accuracy: 83.79%


In [18]:
# Save the model
alexnet_model.save('pneumonia_model_alex.h5')

e:\pneumonia\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [19]:
def preprocess_image(image_path, img_height=224, img_width=224):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=3)
    img = tf.image.resize(img, [img_height, img_width])
    img /= 255.0  # Normalize to [0,1] range
    img = tf.expand_dims(img, 0)  # Add batch dimension
    return img

In [20]:
def predict_image_class(model, image_path):
    img = preprocess_image(image_path)
    predictions = model.predict(img)
    predicted_class = np.argmax(predictions, axis=1)[0]  # Get the index of the max probability
    confidence = np.max(tf.nn.softmax(predictions, axis=1)[0])  # Get the max probability as confidence
    return class_names[predicted_class], confidence

In [22]:
image_path = 'dataset/PNEUMONIA/person1_bacteria_1.jpeg'
predicted_class, confidence = predict_image_class(alexnet_model, image_path)
print(f"Predicted class: {predicted_class} with confidence {confidence:.4f}")

1/1 [==============================] - 0s 112ms/step
Predicted class: PNEUMONIA with confidence 0.8894


In [23]:
image_path = 'dataset/NORMAL/IM-0001-0001.jpeg'
predicted_class, confidence = predict_image_class(alexnet_model, image_path)
print(f"Predicted class: {predicted_class} with confidence {confidence:.4f}")

1/1 [==============================] - 0s 29ms/step
Predicted class: NORMAL with confidence 0.5677


Testing saved model

In [24]:
loaded_model = tf.keras.models.load_model('pneumonia_model_alex.h5')

In [25]:
class_names = ['NORMAL', 'PNEUMONIA']

def preprocess_image(image_path, img_height=224, img_width=224):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [img_height, img_width])
    img /= 255.0  # Normalize to [0,1] range
    img = tf.expand_dims(img, 0)  # Add batch dimension
    return img

def predict_image_class(loaded_model, image_path):
    img = preprocess_image(image_path)
    predictions = loaded_model.predict(img)
    predicted_class = np.argmax(predictions, axis=1)[0]  # Get the index of the max probability
    confidence = np.max(tf.nn.softmax(predictions, axis=1)[0])  # Get the max probability as confidence
    return class_names[predicted_class], confidence

image_path = 'dataset/PNEUMONIA/person1_bacteria_1.jpeg' 

# Use the loaded model to predict the image class
predicted_class, confidence = predict_image_class(loaded_model, image_path)

# Print the prediction result
print(f"Predicted class: {predicted_class} with confidence {confidence:.4f}")

1/1 [==============================] - 0s 93ms/step
Predicted class: PNEUMONIA with confidence 0.8894
